# 05 - Evaluación final, umbral de decisión e interpretación

**Proyecto:** Predicción de ausentismo a turnos médicos (*No-Show*)
**Fase CRISP-DM:** Evaluación (y preparación para el despliegue)

**Objetivo:** ajustar el umbral de decisión del modelo seleccionado en el notebook 04, evaluarlo **una única vez** en el conjunto de prueba, interpretar qué factores explican el ausentismo y cuantificar su utilidad práctica para un centro de salud.

**Experimento registrado en MLflow:** `04_modelo_final`. El modelo final se registra además en el *Model Registry* de MLflow.

**Salidas:** figuras (prefijo `fin_`), tablas en `reports/tables/`, modelo serializado en `models/` y resumen en `reports/tables/fin_resumen.md`.

> **Regla central de este notebook:** el conjunto de prueba no participa de ninguna decisión. El umbral se elige con las predicciones fuera de fold del entrenamiento y todos los modelos evaluados en prueba quedaron definidos en los notebooks anteriores.

## 0. Configuración

In [ ]:
# Interruptor de datos: True = muestra de desarrollo, False = dataset completo
USAR_MUESTRA = False

# Réplicas del bootstrap para los intervalos de confianza en prueba
N_BOOTSTRAP = 1000

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import contextlib
import hashlib
import io
import json
import logging
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import seaborn as sns
from mlflow.models import infer_signature
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve
from sklearn.tree import export_text, plot_tree

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src import config
from src.data import cargar_procesados, ruta_procesados
from src.evaluation import (NOMBRES_METRICAS, bootstrap_por_grupo, evaluar_cv, metricas_ponderadas,
                            umbral_f1_optimo, umbral_para_capacidad, umbral_para_recall)
from src.features import GRUPO
from src.modeling import ESTRATEGIAS, MODELOS, atributos_de, crear_pipeline
from src.reporting import guardar_figura, guardar_tabla, tabla_markdown
from src.tracking import configurar_experimento, etiquetas_base, registrar_corrida

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold"})
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pct = plt.FuncFormatter(lambda v, _: f"{v:.0%}")

resumen = {"Datos utilizados": "Muestra de desarrollo" if USAR_MUESTRA else "Dataset completo"}

## 1. Carga de los datos, controles y configuración seleccionada

In [ ]:
ruta = ruta_procesados(USAR_MUESTRA)
etiquetas = etiquetas_base(USAR_MUESTRA)
assert hashlib.sha256(ruta.read_bytes()).hexdigest() == etiquetas["dataset_sha256"], "El dataset no coincide con los metadatos"

ruta_sel = config.TABLES_DIR / ("opt_seleccion_muestra.json" if USAR_MUESTRA else "opt_seleccion.json")
seleccion = json.loads(ruta_sel.read_text(encoding="utf-8"))
assert seleccion["dataset_sha256"] == etiquetas["dataset_sha256"], "La selección se hizo con otro dataset"

datos = cargar_procesados(USAR_MUESTRA)
train = datos[datos["particion"] == "train"].reset_index(drop=True)
test = datos[datos["particion"] == "test"].reset_index(drop=True)
y_train, y_test = train[config.TARGET], test[config.TARGET]
folds = train["fold"].to_numpy()
ratio_clases = (y_train == 0).sum() / (y_train == 1).sum()
assert not set(train[GRUPO]) & set(test[GRUPO]), "Hay pacientes en ambos conjuntos"

final = seleccion["modelo_final"]
arbol_ref = seleccion["referencia_interpretable"]
resumen["Turnos de entrenamiento / prueba"] = f"{len(train):,} / {len(test):,}"
resumen["Tasa de no-show en prueba"] = f"{y_test.mean():.2%}"
print(f"Entrenamiento: {len(train):,} | Prueba: {len(test):,} turnos ({y_test.mean():.2%} no-show)")
print(f"Modelo final: {MODELOS[final['modelo']][1]} ({final['version']}), PR-AUC CV = {final['pr_auc_cv']:.4f}")
print(f"Referencia interpretable: {MODELOS[arbol_ref['modelo']][1]} ({arbol_ref['version']}), PR-AUC CV = {arbol_ref['pr_auc_cv']:.4f}")

### 1.1. Modelos que se evalúan en prueba

Todos los modelos de esta tabla quedaron definidos antes de mirar el conjunto de prueba. Solo el modelo final es la "respuesta" del trabajo; los demás son referencias que ayudan a interpretar su desempeño.

| Modelo | Rol |
|---|---|
| **Modelo final** | Seleccionado en el notebook 04 (mayor PR-AUC en validación cruzada) |
| Dummy (clase a priori) | Piso sin información |
| Modelo final con parámetros por defecto | Mide cuánto aportó la optimización fuera de la validación cruzada |
| Árbol de Decisión optimizado | Referencia interpretable seleccionada en el notebook 04 |
| Árbol de Decisión de profundidad 3 | Modelo de reglas simples para explicar los patrones (sección 6) |

In [ ]:
def configuracion(modelo, estrategia, params):
    return {"modelo": modelo, "estrategia": estrategia, "params": params}


PROFUNDIDAD_REGLAS = 3
modelos_eval = {
    "final": configuracion(final["modelo"], final["estrategia"], final["params"]),
    "dummy": configuracion("dummy", "ninguna", {}),
    "final_por_defecto": configuracion(final["modelo"], "ninguna", {}),
    "arbol_optimizado": configuracion("arbol", arbol_ref["estrategia"], arbol_ref["params"]),
    "arbol_reglas": configuracion("arbol", "ninguna", {"max_depth": PROFUNDIDAD_REGLAS,
                                                       "min_samples_leaf": arbol_ref["params"].get("min_samples_leaf", 100)}),
}
NOMBRES = {
    "final": f"{MODELOS[final['modelo']][1]} {'optimizado' if final['version'] == 'optimizado' else 'por defecto'} (final)",
    "dummy": "Dummy (clase a priori)",
    "final_por_defecto": f"{MODELOS[final['modelo']][1]} por defecto",
    "arbol_optimizado": "Árbol de Decisión optimizado",
    "arbol_reglas": f"Árbol de Decisión (profundidad {PROFUNDIDAD_REGLAS})",
}
pd.DataFrame([{"Clave": k, "Modelo": NOMBRES[k], "Estrategia": ESTRATEGIAS[v["estrategia"]],
               "Hiperparámetros": ", ".join(f"{a}={b:.4g}" if isinstance(b, float) else f"{a}={b}" for a, b in v["params"].items()) or "por defecto"}
              for k, v in modelos_eval.items()])

## 2. Ajuste del umbral de decisión (solo entrenamiento)

El notebook 03 mostró que el desbalance de clases se resuelve con el **umbral**, no remuestreando los datos. Aquí se elige ese umbral con las **predicciones fuera de fold** del entrenamiento: cada turno recibe una probabilidad de un modelo que no lo vio, por lo que el umbral se ajusta sobre datos "no vistos" sin tocar la prueba.

### 2.1. Predicciones fuera de fold

In [ ]:
oof, cv = {}, {}
for clave, cfg in modelos_eval.items():
    pipeline = crear_pipeline(cfg["modelo"], cfg["estrategia"], ratio_clases, **cfg["params"])
    por_fold, oof[clave] = evaluar_cv(pipeline, train[atributos_de(cfg["modelo"])], y_train, folds)
    cv[clave] = por_fold

diferencia = abs(cv["final"]["pr_auc"].mean() - final["pr_auc_cv"])
assert diferencia < 1e-6, f"La PR-AUC del modelo final no coincide con el notebook 04 ({diferencia:.2e})"
print(f"PR-AUC CV del modelo final: {cv['final']['pr_auc'].mean():.4f} (coincide con el notebook 04)")
pd.DataFrame({NOMBRES[k]: {"PR-AUC (CV)": v["pr_auc"].mean(), "ROC-AUC (CV)": v["roc_auc"].mean()} for k, v in cv.items()}).T

### 2.2. Puntos de operación

El umbral adecuado depende de cómo se use el modelo. Se analizan cuatro criterios:

| Criterio | Pregunta que responde |
|---|---|
| **Máximo F1** (criterio principal) | ¿Qué umbral equilibra mejor la detección de ausencias (recall) y la proporción de alertas correctas (precisión)? |
| Recall mínimo del 50% / 70% | Si el centro quiere detectar al menos esa proporción de las ausencias, ¿con qué precisión puede hacerlo? |
| Capacidad del 20% | Si el centro solo puede contactar al 20% de los turnos, ¿cuántas ausencias detecta contactando a los de mayor riesgo? |

Se adopta el **máximo F1** como umbral principal porque no supone una capacidad operativa ni un costo relativo de errores que el trabajo no conoce. Los otros puntos se informan para que el centro de salud pueda elegir según sus recursos.

In [ ]:
p_oof, y_oof = oof["final"], y_train.to_numpy()
umbral_f1, _ = umbral_f1_optimo(y_oof, p_oof)
puntos = {
    "Máximo F1 (principal)": umbral_f1,
    "Recall mínimo 50%": umbral_para_recall(y_oof, p_oof, 0.50),
    "Recall mínimo 70%": umbral_para_recall(y_oof, p_oof, 0.70),
    "Capacidad 20% de los turnos": umbral_para_capacidad(p_oof, 0.20),
}
filas = []
for nombre, u in puntos.items():
    m = metricas_ponderadas(y_oof, p_oof, u)
    filas.append({"Punto de operación": nombre, "Umbral": u, "Recall": m["recall"], "Precisión": m["precision"],
                  "F1": m["f1"], "% de turnos alertados": (p_oof >= u).mean() * 100})
puntos_oof = pd.DataFrame(filas)
guardar_tabla(puntos_oof, "fin_puntos_operacion_oof")
UMBRAL = umbral_f1
resumen["Umbral de decisión (máximo F1 en OOF)"] = f"{UMBRAL:.4f}"
puntos_oof

In [ ]:
precision, recall, umbrales = precision_recall_curve(y_oof, p_oof)
f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(umbrales, precision[:-1], label="Precisión (no-show)", color="#4C72B0")
ax.plot(umbrales, recall[:-1], label="Recall (no-show)", color="#DD8452")
ax.plot(umbrales, f1[:-1], label="F1 (no-show)", color="#55A868", lw=2.2)
ax.axvline(0.5, color="gray", ls=":", lw=1, label="Umbral por defecto (0,5)")
ax.axvline(UMBRAL, color="black", ls="--", lw=1.4, label=f"Umbral elegido ({UMBRAL:.3f})")
ax.set_xlabel("Umbral de decisión")
ax.set_ylabel("Valor de la métrica")
ax.set_xlim(0, min(1, np.quantile(p_oof, 0.999) + 0.05))
ax.set_title("Figura 25. Precisión, recall y F1 según el umbral (predicciones fuera de fold)")
ax.legend(fontsize=8, loc="center right")
guardar_figura(fig, "fin_01_umbral")
plt.show()

m05 = metricas_ponderadas(y_oof, p_oof, 0.5)
mu = metricas_ponderadas(y_oof, p_oof, UMBRAL)
print(f"Con umbral 0,5:   recall = {m05['recall']:.3f}, precisión = {m05['precision']:.3f}, F1 = {m05['f1']:.3f}")
print(f"Con umbral {UMBRAL:.3f}: recall = {mu['recall']:.3f}, precisión = {mu['precision']:.3f}, F1 = {mu['f1']:.3f}")

**Lectura:** con el umbral por defecto de 0,5 el modelo casi nunca predice "no asiste", por el desbalance de clases. Al bajar el umbral hasta el máximo F1, el recall aumenta mucho a cambio de una precisión menor. Este es el ajuste que reemplaza al remuestreo de los datos, tal como se fundamentó en el notebook 03.

## 3. Entrenamiento final y evaluación única en el conjunto de prueba

Cada modelo se entrena con **todo** el conjunto de entrenamiento y se evalúa una sola vez en prueba. Los umbrales de cada modelo se eligen con el mismo criterio (máximo F1 en sus predicciones fuera de fold), para que la comparación sea justa.

Los intervalos de confianza del 95% se obtienen con un **bootstrap por paciente**: en cada réplica se remuestrean pacientes (no turnos) con reposición, respetando que los turnos de un mismo paciente no son independientes.

In [ ]:
entrenados, proba_test, umbrales_modelo = {}, {}, {}
for clave, cfg in modelos_eval.items():
    columnas = atributos_de(cfg["modelo"])
    entrenados[clave] = crear_pipeline(cfg["modelo"], cfg["estrategia"], ratio_clases, **cfg["params"]).fit(
        train[columnas], y_train)
    proba_test[clave] = entrenados[clave].predict_proba(test[columnas])[:, 1]
    umbrales_modelo[clave] = umbral_f1_optimo(y_oof, oof[clave])[0] if clave != "dummy" else 0.5
umbrales_modelo["final"] = UMBRAL

filas = []
for clave in modelos_eval:
    m = metricas_ponderadas(y_test.to_numpy(), proba_test[clave], umbrales_modelo[clave])
    filas.append({"Modelo": NOMBRES[clave], "Umbral": umbrales_modelo[clave],
                  **{NOMBRES_METRICAS[k]: v for k, v in m.items()}, "PR-AUC (CV)": cv[clave]["pr_auc"].mean()})
resultados_test = pd.DataFrame(filas)
guardar_tabla(resultados_test, "fin_resultados_prueba")
resultados_test

In [ ]:
claves_ic = [k for k in modelos_eval if k != "dummy"]
boot = bootstrap_por_grupo(y_test, {k: proba_test[k] for k in claves_ic}, {k: umbrales_modelo[k] for k in claves_ic},
                           test[GRUPO], n=N_BOOTSTRAP, seed=config.SEED)

ic = (boot.groupby(["modelo", "metrica"])["valor"].quantile([0.025, 0.975]).unstack()
      .rename(columns={0.025: "ic_inf", 0.975: "ic_sup"}).reset_index())
estimacion = pd.DataFrame([{"modelo": k, "metrica": met, "valor": v}
                           for k in claves_ic
                           for met, v in metricas_ponderadas(y_test.to_numpy(), proba_test[k], umbrales_modelo[k]).items()])
tabla_ic = estimacion.merge(ic, on=["modelo", "metrica"])
tabla_ic["Modelo"] = tabla_ic["modelo"].map(NOMBRES)
tabla_ic["Métrica"] = tabla_ic["metrica"].map(NOMBRES_METRICAS)
tabla_ic["Valor (IC 95%)"] = tabla_ic.apply(lambda r: f"{r['valor']:.4f} ({r['ic_inf']:.4f} - {r['ic_sup']:.4f})", axis=1)
tabla_ic_ancha = tabla_ic.pivot(index="Modelo", columns="Métrica", values="Valor (IC 95%)")
tabla_ic_ancha = tabla_ic_ancha[[NOMBRES_METRICAS[m] for m in ["pr_auc", "roc_auc", "recall", "precision", "f1", "brier"]]]
tabla_ic_ancha = tabla_ic_ancha.loc[[NOMBRES[k] for k in claves_ic]]
guardar_tabla(tabla_ic_ancha, "fin_resultados_prueba_ic", index=True)
tabla_ic_ancha

### 3.1. Diferencias pareadas en prueba

Como todos los modelos se evalúan sobre las mismas réplicas del bootstrap, se puede calcular el intervalo de la **diferencia** de PR-AUC entre el modelo final y cada referencia. Si el intervalo no incluye el cero, la diferencia es estadísticamente consistente.

In [ ]:
pr = boot[boot["metrica"] == "pr_auc"].pivot(index="replica", columns="modelo", values="valor")
filas = []
for clave in [k for k in claves_ic if k != "final"]:
    dif = pr["final"] - pr[clave]
    filas.append({"Comparación": f"Final - {NOMBRES[clave]}",
                  "Δ PR-AUC (prueba)": (metricas_ponderadas(y_test.to_numpy(), proba_test["final"], UMBRAL)["pr_auc"]
                                        - metricas_ponderadas(y_test.to_numpy(), proba_test[clave], UMBRAL)["pr_auc"]),
                  "IC 95% inferior": dif.quantile(0.025), "IC 95% superior": dif.quantile(0.975),
                  "Réplicas en que el final es mejor": f"{(dif > 0).mean():.1%}"})
diferencias_test = pd.DataFrame(filas)
diferencias_test["¿Diferencia consistente?"] = np.where(diferencias_test["IC 95% inferior"] > 0, "Sí", "No")
guardar_tabla(diferencias_test, "fin_diferencias_prueba")
diferencias_test

### 3.2. Validación cruzada frente a prueba

Si la PR-AUC de prueba es similar a la de validación cruzada, el modelo generaliza y el sesgo de selección de la optimización fue pequeño. Una caída grande indicaría sobreajuste a los folds de entrenamiento.

In [ ]:
cv_vs_test = pd.DataFrame([{"Modelo": NOMBRES[k], "PR-AUC (CV)": cv[k]["pr_auc"].mean(),
                            "PR-AUC (CV, desvío)": cv[k]["pr_auc"].std(),
                            "PR-AUC (prueba)": metricas_ponderadas(y_test.to_numpy(), proba_test[k], 0.5)["pr_auc"]}
                           for k in modelos_eval])
cv_vs_test["Diferencia (prueba - CV)"] = cv_vs_test["PR-AUC (prueba)"] - cv_vs_test["PR-AUC (CV)"]
guardar_tabla(cv_vs_test, "fin_cv_vs_prueba")

m_final = metricas_ponderadas(y_test.to_numpy(), proba_test["final"], UMBRAL)
ic_final = tabla_ic[tabla_ic["modelo"] == "final"].set_index("metrica")
resumen["PR-AUC en prueba (IC 95%)"] = f"{m_final['pr_auc']:.4f} ({ic_final.loc['pr_auc', 'ic_inf']:.4f} - {ic_final.loc['pr_auc', 'ic_sup']:.4f})"
resumen["ROC-AUC en prueba (IC 95%)"] = f"{m_final['roc_auc']:.4f} ({ic_final.loc['roc_auc', 'ic_inf']:.4f} - {ic_final.loc['roc_auc', 'ic_sup']:.4f})"
resumen["Recall / precisión / F1 en prueba (umbral elegido)"] = f"{m_final['recall']:.3f} / {m_final['precision']:.3f} / {m_final['f1']:.3f}"
resumen["PR-AUC en validación cruzada"] = f"{cv['final']['pr_auc'].mean():.4f} ± {cv['final']['pr_auc'].std():.4f}"
resumen["Piso sin información (tasa de no-show en prueba)"] = f"{y_test.mean():.4f}"
cv_vs_test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
colores = dict(zip(modelos_eval, ["#C44E52", "#8C8C8C", "#4C72B0", "#55A868", "#8172B2"]))
for clave in [k for k in modelos_eval if k != "dummy"]:  # el Dummy se representa con la línea del piso
    prec, rec, _ = precision_recall_curve(y_test, proba_test[clave])
    fpr, tpr, _ = roc_curve(y_test, proba_test[clave])
    m = metricas_ponderadas(y_test.to_numpy(), proba_test[clave], 0.5)
    ancho = 2.6 if clave == "final" else 1.4
    axes[0].plot(rec, prec, color=colores[clave], lw=ancho, label=f"{NOMBRES[clave]} ({m['pr_auc']:.3f})")
    axes[1].plot(fpr, tpr, color=colores[clave], lw=ancho, label=f"{NOMBRES[clave]} ({m['roc_auc']:.3f})")
pf = metricas_ponderadas(y_test.to_numpy(), proba_test["final"], UMBRAL)
axes[0].scatter(pf["recall"], pf["precision"], s=80, color="black", zorder=5, label=f"Umbral elegido ({UMBRAL:.3f})")
axes[0].axhline(y_test.mean(), color="gray", ls="--", lw=1, label=f"Piso sin información ({y_test.mean():.3f})")
axes[0].set(xlabel="Recall (no-show)", ylabel="Precisión (no-show)", title="Curvas precisión-recall (prueba)", ylim=(0, 1))
axes[1].plot([0, 1], [0, 1], color="gray", ls="--", lw=1, label="Piso sin información (0,500)")
axes[1].set(xlabel="Tasa de falsos positivos", ylabel="Tasa de verdaderos positivos", title="Curvas ROC (prueba)")
for ax in axes:
    ax.legend(fontsize=7.5, loc="upper right" if ax is axes[0] else "lower right")
fig.suptitle("Figura 26. Desempeño en el conjunto de prueba", fontweight="bold", y=1.01)
fig.tight_layout()
guardar_figura(fig, "fin_02_curvas_prueba")
plt.show()

## 4. Desempeño operativo del modelo final

### 4.1. Matriz de confusión con el umbral elegido

In [ ]:
y_pred = (proba_test["final"] >= UMBRAL).astype(int)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
etiquetas_cm = ["Asiste", "No asiste"]
anot = np.array([[f"{v:,}".replace(",", ".") + f"\n({v / cm.sum():.1%})" for v in fila] for fila in cm])
sns.heatmap(cm, annot=anot, fmt="", cmap="Blues", cbar=False, xticklabels=etiquetas_cm, yticklabels=etiquetas_cm, ax=axes[0])
axes[0].set(xlabel="Predicción", ylabel="Valor real", title=f"Umbral elegido ({UMBRAL:.3f})")

cm05 = confusion_matrix(y_test, (proba_test["final"] >= 0.5).astype(int))
anot05 = np.array([[f"{v:,}".replace(",", ".") + f"\n({v / cm05.sum():.1%})" for v in fila] for fila in cm05])
sns.heatmap(cm05, annot=anot05, fmt="", cmap="Greys", cbar=False, xticklabels=etiquetas_cm, yticklabels=etiquetas_cm, ax=axes[1])
axes[1].set(xlabel="Predicción", ylabel="Valor real", title="Umbral por defecto (0,5)")
fig.suptitle("Figura 27. Matriz de confusión del modelo final en prueba", fontweight="bold", y=1.02)
fig.tight_layout()
guardar_figura(fig, "fin_03_matriz_confusion")
plt.show()

por_mil = pd.DataFrame({
    "Concepto": ["Turnos alertados como riesgo", "Ausencias detectadas (verdaderos positivos)",
                 "Falsas alarmas (asisten pese a la alerta)", "Ausencias no detectadas (falsos negativos)",
                 "Ausencias totales"],
    "Cada 1.000 turnos": np.round(np.array([tp + fp, tp, fp, fn, tp + fn]) / len(test) * 1000).astype(int),
})
guardar_tabla(por_mil, "fin_impacto_por_mil")
resumen["Cada 1.000 turnos: alertas / ausencias detectadas / ausencias totales"] = (
    f"{por_mil.iloc[0, 1]} / {por_mil.iloc[1, 1]} / {por_mil.iloc[4, 1]}")
por_mil

### 4.2. Curva de ganancia: ¿cuánto rinde contactar a los turnos de mayor riesgo?

Se ordenan los turnos de prueba de mayor a menor probabilidad de ausencia y se calcula qué proporción de las ausencias reales se concentra en cada porcentaje de turnos contactados. Un modelo sin información (contactar al azar) captura una proporción de ausencias igual a la de turnos contactados (diagonal).

In [ ]:
orden = np.argsort(-proba_test["final"])
capturadas = np.cumsum(y_test.to_numpy()[orden]) / y_test.sum()
fraccion = np.arange(1, len(orden) + 1) / len(orden)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(fraccion, capturadas, color="#C44E52", lw=2.4, label="Modelo final")
ax.plot([0, 1], [0, 1], color="gray", ls="--", lw=1, label="Contacto al azar")
filas = []
for f in [0.1, 0.2, 0.3, 0.5]:
    c = capturadas[int(f * len(orden)) - 1]
    ax.scatter(f, c, color="black", zorder=5, s=25)
    ax.annotate(f"{f:.0%} → {c:.0%}", (f, c), textcoords="offset points", xytext=(8, -12), fontsize=9)
    filas.append({"% de turnos contactados (mayor riesgo)": f"{f:.0%}", "% de ausencias capturadas": c * 100,
                  "Lift (respecto del azar)": c / f})
alertados = (proba_test["final"] >= UMBRAL).mean()
ax.axvline(alertados, color="black", ls=":", lw=1, label=f"Turnos alertados con el umbral ({alertados:.0%})")
ax.xaxis.set_major_formatter(pct)
ax.yaxis.set_major_formatter(pct)
ax.set(xlabel="Turnos contactados, ordenados por riesgo", ylabel="Ausencias capturadas",
       title="Figura 28. Curva de ganancia del modelo final (prueba)")
ax.legend(loc="lower right")
guardar_figura(fig, "fin_04_curva_ganancia")
plt.show()

ganancia = pd.DataFrame(filas)
guardar_tabla(ganancia, "fin_curva_ganancia")
resumen["Contactando al 20% de mayor riesgo se captura"] = f"{ganancia.iloc[1, 1]:.1f}% de las ausencias (lift {ganancia.iloc[1, 2]:.2f})"
ganancia

### 4.3. Calibración de las probabilidades

Un modelo está bien calibrado si, entre los turnos a los que asigna una probabilidad de ausencia del 30%, falta aproximadamente el 30%. La calibración importa si el centro quiere usar la probabilidad en sí (por ejemplo, para estimar cuántos sobreturnos asignar), no solo el orden de riesgo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.2]})
for clave in ["final", "arbol_optimizado"]:
    frac_pos, prob_media = calibration_curve(y_test, proba_test[clave], n_bins=10, strategy="quantile")
    axes[0].plot(prob_media, frac_pos, marker="o", color=colores[clave], label=NOMBRES[clave])
axes[0].plot([0, 1], [0, 1], color="gray", ls="--", lw=1, label="Calibración perfecta")
lim = max(0.7, float(np.quantile(proba_test["final"], 0.999)) + 0.05)
axes[0].set(xlim=(0, lim), ylim=(0, lim), xlabel="Probabilidad predicha (media por decil)",
            ylabel="Proporción real de ausencias", title="Curva de calibración (prueba)")
axes[0].legend(fontsize=8)
for valor, color, nombre in [(0, "#4C72B0", "Asiste"), (1, "#DD8452", "No asiste")]:
    axes[1].hist(proba_test["final"][y_test.to_numpy() == valor], bins=40, alpha=0.6, color=color, label=nombre, density=True)
axes[1].axvline(UMBRAL, color="black", ls="--", lw=1.2, label=f"Umbral ({UMBRAL:.3f})")
axes[1].set(xlabel="Probabilidad de ausencia predicha", ylabel="Densidad",
            title="Distribución de probabilidades del modelo final")
axes[1].legend(fontsize=8)
fig.suptitle("Figura 29. Calibración y separación de clases", fontweight="bold", y=1.02)
fig.tight_layout()
guardar_figura(fig, "fin_05_calibracion")
plt.show()
print(f"Brier del modelo final en prueba: {m_final['brier']:.4f} | Brier de la clase a priori: "
      f"{metricas_ponderadas(y_test.to_numpy(), proba_test['dummy'], 0.5)['brier']:.4f}")

## 5. ¿Qué atributos explican el ausentismo? Importancia por permutación

Para cada atributo se mezclan al azar sus valores en el conjunto de prueba y se mide cuánto cae la PR-AUC del modelo final. Si la caída es grande, el modelo depende de ese atributo. Se repite 10 veces para estimar la variabilidad.

**Precaución de interpretación:** cuando dos atributos comparten información (por ejemplo, `anticipacion_dias` y `mismo_dia`, o los atributos de historial entre sí), permutar uno solo subestima su importancia, porque el modelo puede apoyarse en el otro. Por eso la importancia se analiza también por **grupos de atributos**.

In [ ]:
columnas_final = atributos_de(final["modelo"])
perm = permutation_importance(entrenados["final"], test[columnas_final], y_test, scoring="average_precision",
                              n_repeats=10, random_state=config.SEED, n_jobs=1)
imp = pd.DataFrame({"Atributo": columnas_final, "Caída de PR-AUC (media)": perm.importances_mean,
                    "Caída de PR-AUC (desvío)": perm.importances_std}).sort_values("Caída de PR-AUC (media)", ascending=False)
guardar_tabla(imp, "fin_importancia_permutacion")

GRUPOS_ATRIBUTOS = {
    "Anticipación del turno": ["anticipacion_dias", "mismo_dia"],
    "Historial del paciente": ["turnos_previos", "ausencias_previas", "tasa_ausencias_previas", "tiene_historial"],
    "Agenda del día": ["turnos_mismo_dia", "dia_semana", "hora_agenda"],
    "Recordatorio SMS": ["sms_recibido"],
    "Demográficos": ["edad", "grupo_edad", "mujer", "barrio", "beca"],
    "Clínicos": ["hipertension", "diabetes", "alcoholismo", "discapacidad"],
}
rng = np.random.default_rng(config.SEED)
base = metricas_ponderadas(y_test.to_numpy(), proba_test["final"], UMBRAL)["pr_auc"]
filas = []
for grupo, cols in GRUPOS_ATRIBUTOS.items():
    cols = [c for c in cols if c in columnas_final]  # solo los atributos que usa el modelo final
    if not cols:
        continue
    caidas = []
    for _ in range(10):
        X_perm = test[columnas_final].copy()
        indice = rng.permutation(len(X_perm))
        X_perm[cols] = X_perm[cols].to_numpy()[indice]  # se permutan juntos: conserva su relación interna
        caidas.append(base - metricas_ponderadas(y_test.to_numpy(), entrenados["final"].predict_proba(X_perm)[:, 1], UMBRAL)["pr_auc"])
    filas.append({"Grupo": grupo, "Atributos": ", ".join(cols), "Caída de PR-AUC (media)": np.mean(caidas),
                  "Caída de PR-AUC (desvío)": np.std(caidas)})
imp_grupos = pd.DataFrame(filas).sort_values("Caída de PR-AUC (media)", ascending=False)
guardar_tabla(imp_grupos, "fin_importancia_grupos")

fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.1, 1]})
sub = imp.iloc[::-1]
axes[0].barh(sub["Atributo"], sub["Caída de PR-AUC (media)"], xerr=sub["Caída de PR-AUC (desvío)"], color="#4C72B0", capsize=2)
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set(xlabel="Caída de PR-AUC al permutar", title="Por atributo")
sub = imp_grupos.iloc[::-1]
axes[1].barh(sub["Grupo"], sub["Caída de PR-AUC (media)"], xerr=sub["Caída de PR-AUC (desvío)"], color="#DD8452", capsize=3)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set(xlabel="Caída de PR-AUC al permutar el grupo completo", title="Por grupo de atributos")
fig.suptitle("Figura 30. Importancia por permutación del modelo final (prueba)", fontweight="bold", y=1.01)
fig.tight_layout()
guardar_figura(fig, "fin_06_importancia_permutacion")
plt.show()

resumen["Grupo de atributos más importante"] = f"{imp_grupos.iloc[0]['Grupo']} (caída de PR-AUC {imp_grupos.iloc[0]['Caída de PR-AUC (media)']:.4f})"
imp_grupos

**Lectura:** la comparación entre ambos paneles es tan informativa como el ranking. Si la importancia de un grupo es **menor** que la suma de las importancias individuales de sus atributos, la diferencia se debe a un artefacto de la permutación individual: al mezclar un atributo solo se generan combinaciones imposibles (por ejemplo, más ausencias previas que turnos previos, o `mismo_dia = 1` con varios días de anticipación) y el modelo responde a esos casos irreales, lo que **exagera** la importancia de cada atributo por separado. Al permutar el grupo completo, sus atributos conservan la coherencia entre sí, por lo que la importancia por grupos es la medida más confiable para concluir qué tipo de información explica el ausentismo.

## 6. Reglas interpretables: árbol de decisión de profundidad 3

El árbol optimizado del notebook 04 tiene muchos niveles: es preciso, pero sus reglas no se pueden leer. Para explicar los patrones se entrena un árbol de **profundidad 3** (como máximo 8 reglas), con el mismo tamaño mínimo de hoja que el árbol optimizado. Su desempeño, informado en la sección 3, muestra cuánto se resigna a cambio de simplicidad.

En el árbol, el atributo `barrio` aparece **codificado por objetivo**: su valor es la tasa de no-show (suavizada) del barrio en el entrenamiento.

In [ ]:
arbol = entrenados["arbol_reglas"].named_steps["clasificador"]
nombres_atributos = list(entrenados["arbol_reglas"].named_steps["preprocesador"].get_feature_names_out())

fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(arbol, feature_names=nombres_atributos, class_names=["Asiste", "No asiste"], filled=True, rounded=True,
          proportion=True, precision=2, impurity=False, fontsize=9, ax=ax)
ax.set_title(f"Figura 31. Árbol de decisión de profundidad {PROFUNDIDAD_REGLAS} (proporciones de cada nodo)",
             fontweight="bold", fontsize=14)
guardar_figura(fig, "fin_07_arbol_reglas")
plt.show()

In [ ]:
# Reglas en texto y tabla de hojas: condición, cantidad de turnos y tasa de no-show
print(export_text(arbol, feature_names=nombres_atributos, decimals=2))

X_tr_arbol = entrenados["arbol_reglas"].named_steps["preprocesador"].transform(train[atributos_de("arbol")])
X_te_arbol = entrenados["arbol_reglas"].named_steps["preprocesador"].transform(test[atributos_de("arbol")])
hoja_tr, hoja_te = arbol.apply(X_tr_arbol), arbol.apply(X_te_arbol)


def condicion_de_hoja(arbol, hoja, nombres):
    t = arbol.tree_
    padres = {h: (p, "izq") for p, h in enumerate(t.children_left) if h != -1}
    padres.update({h: (p, "der") for p, h in enumerate(t.children_right) if h != -1})
    partes, nodo = [], hoja
    while nodo in padres:
        p, lado = padres[nodo]
        signo = "<=" if lado == "izq" else ">"
        partes.append(f"{nombres[t.feature[p]]} {signo} {t.threshold[p]:.2f}")
        nodo = p
    return " y ".join(reversed(partes))


reglas = pd.DataFrame({"hoja": np.unique(hoja_tr)})
reglas["Regla"] = reglas["hoja"].map(lambda h: condicion_de_hoja(arbol, h, nombres_atributos))
reglas["Turnos (entrenamiento)"] = reglas["hoja"].map(pd.Series(hoja_tr).value_counts())
reglas["No-show (entrenamiento)"] = reglas["hoja"].map(pd.Series(y_train.to_numpy()).groupby(hoja_tr).mean())
reglas["Turnos (prueba)"] = reglas["hoja"].map(pd.Series(hoja_te).value_counts()).fillna(0).astype(int)
reglas["No-show (prueba)"] = reglas["hoja"].map(pd.Series(y_test.to_numpy()).groupby(hoja_te).mean())
reglas = reglas.sort_values("No-show (entrenamiento)", ascending=False).drop(columns="hoja").reset_index(drop=True)
reglas.index = reglas.index + 1
reglas.index.name = "N.º"
guardar_tabla(reglas, "fin_reglas_arbol", index=True)
pd.set_option("display.max_colwidth", 200)
reglas

**Lectura:** cada fila es un perfil de turno definido por pocas condiciones. La tasa de no-show en prueba de cada regla, calculada con datos que el árbol no vio, permite comprobar que los patrones se sostienen fuera del entrenamiento. Las reglas con tasas muy por encima de la global identifican los perfiles hacia los que conviene dirigir las acciones preventivas.

## 7. Registro en MLflow y persistencia del modelo final

Cada modelo evaluado se registra en el experimento `04_modelo_final` con sus métricas de prueba y su umbral. El modelo final se registra además en el **Model Registry** de MLflow con el nombre `noshow_modelo_final`, junto con su firma de entrada y salida, y se guarda una copia en `models/` con sus metadatos (umbral, *hash* del dataset, *commit*). Por último se verifica que el modelo **recargado** desde MLflow produzca exactamente las mismas probabilidades.

In [ ]:
configurar_experimento("final", USAR_MUESTRA, reiniciar=True)
nombre_registro = "noshow_modelo_final" + ("_muestra" if USAR_MUESTRA else "")
run_ids = {}
for clave, cfg in modelos_eval.items():
    with mlflow.start_run(run_name=clave) as corrida:
        m = metricas_ponderadas(y_test.to_numpy(), proba_test[clave], umbrales_modelo[clave])
        metricas = {f"test_{k}": v for k, v in m.items()}
        metricas.update({"cv_pr_auc_media": cv[clave]["pr_auc"].mean(), "umbral": umbrales_modelo[clave]})
        if clave in claves_ic:
            for met in ["pr_auc", "roc_auc", "recall", "precision", "f1"]:
                fila = tabla_ic[(tabla_ic["modelo"] == clave) & (tabla_ic["metrica"] == met)].iloc[0]
                metricas[f"test_{met}_ic_inf"], metricas[f"test_{met}_ic_sup"] = fila["ic_inf"], fila["ic_sup"]
        registrar_corrida(
            nombre_corrida=clave,
            parametros={"modelo": cfg["modelo"], "estrategia": cfg["estrategia"], "rol": NOMBRES[clave],
                        **{f"clf__{k}": v for k, v in cfg["params"].items()}},
            metricas=metricas,
            etiquetas={**etiquetas, "etapa": "final", "tipo": "evaluacion_prueba"},
        )
        if clave == "final":
            X_ejemplo = test[columnas_final].head(5)
            # MLflow imprime mensajes del registro de modelos; se capturan para no ensuciar la salida
            logging.disable(logging.WARNING)
            with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
                info = mlflow.sklearn.log_model(
                    entrenados["final"], artifact_path="modelo", registered_model_name=nombre_registro,
                    signature=infer_signature(X_ejemplo, entrenados["final"].predict_proba(X_ejemplo)[:, 1]),
                    input_example=X_ejemplo,
                )
            logging.disable(logging.NOTSET)
            version_registrada = info.registered_model_version
            for nombre_fig in ["fin_02_curvas_prueba", "fin_03_matriz_confusion", "fin_04_curva_ganancia",
                               "fin_06_importancia_permutacion"]:
                mlflow.log_artifact(str(config.FIGURES_DIR / f"{nombre_fig}.png"), artifact_path="figuras")
            uri_modelo = info.model_uri
        run_ids[clave] = corrida.info.run_id
print(f"Corridas registradas: {len(run_ids)} | Modelo registrado como '{nombre_registro}', versión {version_registrada}")

In [ ]:
# Verificación de reproducibilidad: el modelo recargado desde MLflow debe predecir exactamente lo mismo
logging.disable(logging.WARNING)
recargado = mlflow.sklearn.load_model(uri_modelo)
logging.disable(logging.NOTSET)
diferencia_max = np.abs(recargado.predict_proba(test[columnas_final])[:, 1] - proba_test["final"]).max()
assert diferencia_max < 1e-12, f"El modelo recargado no reproduce las predicciones ({diferencia_max:.2e})"
print(f"Modelo recargado desde MLflow: diferencia máxima en las probabilidades = {diferencia_max:.1e}")

config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
sufijo = "_muestra" if USAR_MUESTRA else ""
joblib.dump(entrenados["final"], config.MODELS_DIR / f"modelo_final{sufijo}.joblib")
metadatos_modelo = {
    "modelo": final["modelo"], "version": final["version"], "estrategia": final["estrategia"], "params": final["params"],
    "umbral": UMBRAL, "criterio_umbral": "máximo F1 en predicciones fuera de fold del entrenamiento",
    "atributos": columnas_final, "dataset_sha256": etiquetas["dataset_sha256"], "git_commit": etiquetas["git_commit"],
    "mlflow_run_id": run_ids["final"], "mlflow_modelo_registrado": nombre_registro,
    "metricas_prueba": {k: float(v) for k, v in m_final.items()},
}
(config.MODELS_DIR / f"modelo_final{sufijo}.json").write_text(json.dumps(metadatos_modelo, indent=2, ensure_ascii=False),
                                                               encoding="utf-8")
print(f"Modelo guardado en: {config.MODELS_DIR / f'modelo_final{sufijo}.joblib'} (con sus metadatos en .json)")

### 7.1. Exportación del resumen

In [ ]:
cifras = pd.DataFrame(list(resumen.items()), columns=["Indicador", "Valor"])
lineas = [
    "# Resumen de la evaluación final",
    "",
    f"**Datos utilizados:** {resumen['Datos utilizados']}",
    "",
    "## Cifras clave",
    "",
    tabla_markdown(cifras),
    "",
    "## Puntos de operación (predicciones fuera de fold del entrenamiento)",
    "",
    tabla_markdown(puntos_oof.round(4)),
    "",
    "## Resultados en prueba",
    "",
    tabla_markdown(resultados_test.round(4)),
    "",
    "## Resultados en prueba con IC 95% (bootstrap por paciente)",
    "",
    tabla_markdown(tabla_ic_ancha, index=True),
    "",
    "## Diferencias pareadas de PR-AUC en prueba",
    "",
    tabla_markdown(diferencias_test.round(4)),
    "",
    "## Validación cruzada frente a prueba",
    "",
    tabla_markdown(cv_vs_test.round(4)),
    "",
    "## Impacto operativo cada 1.000 turnos",
    "",
    tabla_markdown(por_mil),
    "",
    "## Curva de ganancia",
    "",
    tabla_markdown(ganancia.round(2)),
    "",
    "## Importancia por permutación (grupos)",
    "",
    tabla_markdown(imp_grupos.round(4)),
    "",
    "## Importancia por permutación (atributos)",
    "",
    tabla_markdown(imp.round(4)),
    "",
    "## Reglas del árbol de profundidad 3",
    "",
    tabla_markdown(reglas.round(4), index=True),
    "",
]
ruta_resumen = config.TABLES_DIR / "fin_resumen.md"
ruta_resumen.write_text("\n".join(lineas), encoding="utf-8")
print(f"Resumen guardado en: {ruta_resumen}")
cifras